# Grievance Classification — Model Training

Trains the two core AI models for the Grievance Lodging & Tracking system:

1. **Department Classifier** — predicts which government department a complaint belongs to (Water Supply, Roads & Infrastructure, Electricity, Waste Management, Sanitation, Public Transport, Parks, Others)
2. **Priority Classifier** — predicts urgency (Low / Medium / High)

Both use **TF-IDF + Logistic Regression**. Simple, fast to train, easy to explain in a demo, and accurate enough for this dataset size.

Output: `grievance_model.pkl` — a single bundled file consumed by `predict.py`.

## 1. Imports

In [ ]:
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

DATA_PATH = "data/complaints.csv"
MODEL_PATH = "grievance_model.pkl"
RANDOM_STATE = 42


## 2. Load the dataset

The dataset (`data/complaints.csv`) has 3 columns: `text`, `department`, `priority`.
It was built with `data/generate_dataset.py` — 288 labeled examples, 36 per department, spread across Low/Medium/High priority. Feel free to add more real complaints here as the team collects them; the pipeline doesn't need to change.

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=["text", "department", "priority"])
print(f"Loaded {len(df)} labeled complaints.")
print(f"Departments: {sorted(df['department'].unique())}")
print(f"Priorities:  {sorted(df['priority'].unique())}")
df.head()


In [ ]:
df["department"].value_counts()

In [ ]:
df["priority"].value_counts()

## 3. Train the Department Classifier

TF-IDF (unigrams + bigrams) feeding a Logistic Regression classifier, with `class_weight="balanced"` in case the real-world data ends up imbalanced later.

In [ ]:
def train_department_model(df):
    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df["department"],
        test_size=0.2, random_state=RANDOM_STATE, stratify=df["department"]
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            C=5.0,
            class_weight="balanced",
        )),
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    print("=== Department Classifier ===")
    print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
    print(classification_report(y_test, preds, zero_division=0))

    # Refit on the FULL dataset for the final deployed model
    pipeline.fit(df["text"], df["department"])
    return pipeline

department_model = train_department_model(df)


## 4. Train the Priority Classifier

Same architecture, different label column. Priority is inherently fuzzier than department (a burst pipe is obviously High, but "irregular supply" could be Medium or Low depending on context) — hence the slightly lower accuracy below. Good enough for an MVP triage signal; an admin can always override it.

In [ ]:
def train_priority_model(df):
    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df["priority"],
        test_size=0.2, random_state=RANDOM_STATE, stratify=df["priority"]
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.95,
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            C=5.0,
            class_weight="balanced",
        )),
    ])

    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)

    print("=== Priority Classifier ===")
    print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
    print(classification_report(y_test, preds, zero_division=0))

    pipeline.fit(df["text"], df["priority"])
    return pipeline

priority_model = train_priority_model(df)


## 5. Save the model bundle

Both pipelines get saved into **one** pickle file so `predict.py` only has to load a single artifact.

In [ ]:
bundle = {
    "department_model": department_model,
    "priority_model": priority_model,
    "department_labels": sorted(df["department"].unique().tolist()),
    "priority_labels": sorted(df["priority"].unique().tolist()),
}

joblib.dump(bundle, MODEL_PATH)
print(f"Saved trained models to {MODEL_PATH}")


## 6. Quick sanity check

Run a few examples straight from the project spec through both models.

In [ ]:
def quick_predict(text):
    dept = department_model.predict([text])[0]
    conf = max(department_model.predict_proba([text])[0])
    pri = priority_model.predict([text])[0]
    print(f"IN : {text}")
    print(f"OUT: department={dept} (confidence={conf:.2f}), priority={pri}\n")

quick_predict("There has been no water supply in my locality for three days.")
quick_predict("No street lights are working.")
quick_predict("Transformer exploded and wires are sparking.")
quick_predict("There has been no garbage collection for two weeks in Sector 8.")


## Next step

Run `python3 predict.py` to start the Flask API (or import `predict_department`, `predict_priority`, `predict_summary`, `analyze_complaint` directly from `predict.py` in your own scripts). See `README.md` for the full API contract that the Node.js backend should call.